work log: 
Simply speaking, the main target of the assignment is to get data from ElHub using API calls, plot the data in the notebook, and send data to Mongo database, from which it will be later read, processed, and displayed on StreamLit app. Seems easy enough. But! The problems came from docker. to optimize the process, the jupyter otebook, thats responsible for primary data acquisition, transfer. and processing, needed to be run in docker container. Wouldnt be an issue, right? Well, the problem was that everything was set up and code was under development, some errors kept popping up. Firstly, cassandra and jupyter were not able to communicate. The problem was that I, due to lack of experience, made a mistake of putting those two on differet networks. Luckily, the issue was fixed easily with two commands: disconnect from one network and connect to the other. The problems didnt stop on that. The second major issue was that Spark was not willing to work properly. All the time it was complaining on Windows and wanted to install some additional stuff. After many hours and exhaustion of any hopes, the problem was solved with great and mighty ChatGPT. Appears, that even though everything was set up correctly, on one network, with correct versions, when i was running juoyter from VScode it didnt work cause VSCode wasnt connected to docker container. Rookie mistake... As an anxious person, who is not very combortable with installing some random stuff on windows, it was decided to run the notebook locally from browser. It saved me from panicing about installing something, that i dont understand. After that all went smoothly and sound. For future generations and to future me:  
- put all personal info (username/password/etc) to secrets from the beginning
- dont forget to add location of your secrets to yml and every time you change something in yml, dont forget to recreate container. NB! requirement file is very useful, cause each time you will recreate the container, you will have to reinstall libraries
- if you run streamlit online, dont forget to add your secrets to the app settings, otherwise it will complain. 
- chatgpt is a great tool for code generation, but you have to think and analyze yourself. Sometimes is uch easier and faster to find an issue and debug it yourself.

This assignment strengthen my ability to find a problem and solution, even for something that you are not expert in or not fully understand. Great skill in life.

AI usage: chatgpt was used to generate base code and was of help sometimes with debugging.
Links:
- Github notebook :  https://github.com/Viktoria0720/IND320_VY/tree/assignment2/ASSIGNMENTS/PART2/notebooks
- Github streamlit:  https://github.com/Viktoria0720/IND320_VY/tree/assignment2/ASSIGNMENTS/PART1/streamlit_app_1
- Streamlit:  https://ind320vy-app1v2.streamlit.app/

In [1]:
import os, requests, pandas as pd
from datetime import datetime, timezone, timedelta

# === Assignment config ===
YEAR = 2021
CHOSEN_AREA = "NO1"    # used by the plots
USE_MOCK = False       # set True to test full pipeline without calling the API

# === Docker service names / ports ===
CASSANDRA_HOST = os.getenv("CASSANDRA_HOST", "my_cassandra")
CASSANDRA_PORT = int(os.getenv("CASSANDRA_PORT", "9042"))

# === MongoDB Atlas (live cloud) ===
MONGO_URI = os.getenv("MONGO_URI","")

# === Elhub v0 public endpoint (no auth) ===
ELHUB_V0_BASE = "https://api.elhub.no/energy-data/v0"
ENTITY = "price-areas"
DATASET = "PRODUCTION_PER_GROUP_MBA_HOUR"

print(f"Cassandra = {CASSANDRA_HOST}:{CASSANDRA_PORT}")
print(f"Mongo URI set? {'yes' if bool(MONGO_URI) else 'no'}")
print(f"Endpoint = {ELHUB_V0_BASE}/{ENTITY}?dataset={DATASET}")


Cassandra = my_cassandra:9042
Mongo URI set? yes
Endpoint = https://api.elhub.no/energy-data/v0/price-areas?dataset=PRODUCTION_PER_GROUP_MBA_HOUR


In [2]:
# Spark is our engine to write/read Cassandra conveniently.
from pyspark.sql import SparkSession

# Maven coordinate for the Spark–Cassandra connector matching Spark 3.5.x and Scala 2.12
CASSANDRA_CONNECTOR = "com.datastax.spark:spark-cassandra-connector_2.12:3.5.1"

spark = (
    SparkSession.builder
    .appName("IND320-Elhub-v0-Production")
    # Pull connector jar automatically
    .config("spark.jars.packages", CASSANDRA_CONNECTOR)
    # Tell connector where Cassandra lives (container name + port)
    .config("spark.cassandra.connection.host", CASSANDRA_HOST)
    .config("spark.cassandra.connection.port", str(CASSANDRA_PORT))
    # Keep everything in UTC; avoids DST headaches
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)
print("Spark:", spark.version)

# Use Python driver once to create keyspace+table (simpler than doing it via Spark SQL)
from cassandra.cluster import Cluster

cluster = Cluster([CASSANDRA_HOST], port=CASSANDRA_PORT)
session = cluster.connect()

# SimpleStrategy is OK for a single-node dev/test cluster
session.execute("""
CREATE KEYSPACE IF NOT EXISTS elhub
WITH replication = {'class': 'SimpleStrategy', 'replication_factor': '1'}
""")

# Table stores hourly production; primary key lets query by area+group ordered by time
session.execute("""
CREATE TABLE IF NOT EXISTS elhub.production_hourly (
    pricearea text,
    productiongroup text,
    starttime timestamp,
    quantitykwh double,
    PRIMARY KEY ((pricearea, productiongroup), starttime)
) WITH CLUSTERING ORDER BY (starttime ASC)
""")

cluster.shutdown()
print("Cassandra schema ready.")


Spark: 3.5.1
Cassandra schema ready.


In [3]:
import socket, sys
#Connect to cassandra

HOST = "my_cassandra"  
PORT = 9042             

print("Checking DNS…")
try:
    print(socket.gethostbyname_ex(HOST))
except Exception as e:
    print("DNS error:", e)
    sys.exit(0)

print("Checking TCP…")
s = socket.socket()
s.settimeout(3)
try:
    s.connect((HOST, PORT))
    print(f"TCP OK to {HOST}:{PORT}")
finally:
    s.close()


Checking DNS…
('my_cassandra', [], ['172.20.0.2'])
Checking TCP…
TCP OK to my_cassandra:9042


In [ ]:
def hourly_windows_utc(year:int, days_per_chunk:int=7):
    """
    Yield (start,end) UTC windows that cover the chosen year in chunks.
    The public API often limits time span per request, so we chunk by a week.
    """
    start = datetime(year,1,1,tzinfo=timezone.utc)
    end   = datetime(year+1,1,1,tzinfo=timezone.utc)
    step  = timedelta(days=days_per_chunk)
    cur = start
    while cur < end:
        nxt = min(cur+step, end)
        yield (cur, nxt)
        cur = nxt

def fetch_chunk_v0(start_dt:datetime, end_dt:datetime):
    """
    Call the public v0 endpoint for price-areas + dataset=PRODUCTION_PER_GROUP_MBA_HOUR
    between start_dt and end_dt (UTC ISO strings with Z).
    Returns JSON (usually a list of 'price area' dicts).
    """
    url = f"{ELHUB_V0_BASE}/{ENTITY}"
    params = {
        "dataset": DATASET,
        "startTime": start_dt.isoformat().replace("+00:00","Z"),
        "endTime":   end_dt.isoformat().replace("+00:00","Z"),
    }
    r = requests.get(url, params=params, timeout=90)
    r.raise_for_status()
    return r.json()

def extract_price_area_rows(payload):
    """
    v0 typically returns a LIST. Some variants use {'data': [...]}
    We normalize to always return a list.
    """
    if isinstance(payload, list):
        return payload
    if isinstance(payload, dict) and "data" in payload and isinstance(payload["data"], list):
        return payload["data"]
    return []

from datetime import datetime, timezone
from urllib.parse import urlencode

def _iso_utc(dt):
    return dt.astimezone(timezone.utc).isoformat().replace("+00:00", "Z")

def _request_elhub(params):
    url = f"{ELHUB_V0_BASE}/{ENTITY}"
    r = requests.get(url, params=params, headers={"Accept": "application/json"}, timeout=60)
    r.raise_for_status()
    data = r.json()
    if isinstance(data, dict) and "data" in data:
        data = data["data"]
    if not isinstance(data, list):
        raise ValueError(f"Unexpected payload: {type(data)}")
    return data

def fetch_all_2021_v0():
    """
    Force 01.01.2021 00:00Z → 31.12.2021 23:59Z using several parameter conventions.
    If the API still returns other years, we'll filter later.
    """
    start_dt = datetime(2021, 1, 1, 0, 0, tzinfo=timezone.utc)
    end_dt   = datetime(2021, 12, 31, 23, 59, tzinfo=timezone.utc)

    # Try multiple param-name styles used by Elhub variants
    attempts = [
        {"startTimeFrom": _iso_utc(start_dt), "startTimeTo": _iso_utc(end_dt)},
        {"fromTime": _iso_utc(start_dt), "toTime": _iso_utc(end_dt)},
        {"startTime": _iso_utc(start_dt), "endTime": _iso_utc(end_dt)},
    ]

    last_data = None
    for attempt in attempts:
        params = {"dataset": DATASET, **attempt}
        print("Requesting Elhub with params:", params)
        data = _request_elhub(params)
        last_data = data
        # Quick shape check: if any inner hourly item looks in-range, accept this attempt
        try:
            # Peek into one area with data
            for row in data:
                series = row.get("productionPerGroupMbaHour") or row.get("attributes", {}).get("productionPerGroupMbaHour")
                if not series:
                    continue
                # find a timestamp field in the first item
                ts = None
                for k in ("startTime","fromTime","time","datetime","from_time","start"):
                    if k in series[0]:
                        ts = pd.to_datetime(series[0][k], utc=True, errors="coerce")
                        break
                if ts is not None and ts.year == 2021:
                    print("✔ Bounds appear respected for this attempt.")
                    return data
        except Exception:
            pass
        print("…did not look like 2021; trying next param style.")

    # If none validated, return the last payload; we’ll filter it after normalization.
    print("⚠ API ignored all bound variants; will hard-filter after normalization.")
    return last_data or []




In [ ]:
# --- ELHUB 2021 MONTH-BY-MONTH FETCH (v0) ------------------------------------
import time
import requests
import pandas as pd
from datetime import datetime, timedelta, timezone

# Reuse your existing constants if you already have them defined:
# ELHUB_V0_BASE = "https://..."      # your v0 base URL
# ENTITY         = "..."              # e.g. "price-areas"
# DATASET        = "PRODUCTION_PER_GROUP_MBA_HOUR"
# If you don't have ENTITY/BASE set, set BASE_URL directly:
# BASE_URL = "https://.../price-areas"

BASE_URL = f"{ELHUB_V0_BASE}/{ENTITY}"  # comment this line if you set BASE_URL directly above

def format_date(dt: datetime) -> str:
    # UTC ISO8601 with Z
    return dt.astimezone(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

def _normalize_hour_item(area_name: str, item: dict) -> dict | None:
    """Map possible key variants to canonical names; drop placeholders."""
    if not isinstance(item, dict):
        return None
    pg = item.get("productionGroup") or item.get("production_group") or item.get("group") or item.get("groupCode")
    if pg in (None, "*"):
        return None
    # time + quantity variants
    ts = (
        item.get("startTime") or item.get("fromTime") or item.get("time") or
        item.get("datetime") or item.get("start") or item.get("from_time")
    )
    q  = item.get("quantityKwh") or item.get("quantity") or item.get("value") or item.get("kwh") or item.get("amount")
    if ts is None or q is None:
        return None
    return {
        "priceArea": area_name,
        "productionGroup": pg,
        "startTime": ts,
        "quantityKwh": q,
    }

headers = {"Accept": "application/json"}
all_records: list[dict] = []

print("=== Fetching Elhub monthly windows for 2021 ===")
for month in range(1, 13):
    start = datetime(2021, month, 1, tzinfo=timezone.utc)
    # jump >31 days then snap to first of next month (safe for all months)
    next_month = (start + timedelta(days=32)).replace(day=1)
    end = next_month - timedelta(seconds=1)

    start_str = format_date(start)
    end_str   = format_date(end)

    url = f"{BASE_URL}?dataset={DATASET}&startDate={start_str}&endDate={end_str}"
    print(f"\n→ Fetching {start.date()} → {end.date()}")

    try:
        resp = requests.get(url, headers=headers, timeout=60)
        if resp.status_code != 200:
            print(f"  ❌ HTTP {resp.status_code}")
            continue
        data = resp.json()
    except Exception as e:
        print(f"  ❌ Request failed: {e}")
        continue

    # Shape can be either bare list or {"data":[...]}
    if isinstance(data, dict):
        data = data.get("data", [])
    if not isinstance(data, list):
        print(f"  ⚠ Unexpected payload type: {type(data)}")
        continue

    month_rows = 0
    for entry in data:
        attrs = entry.get("attributes", {}) if isinstance(entry, dict) else {}
        area_name = attrs.get("name") or entry.get("name") or attrs.get("priceArea") or "UNKNOWN"
        series = attrs.get("productionPerGroupMbaHour") or entry.get("productionPerGroupMbaHour") or []
        if not isinstance(series, list):
            continue

        for item in series:
            rec = _normalize_hour_item(area_name, item)
            if rec is not None:
                all_records.append(rec)
                month_rows += 1

    print(f"  ✅ Added {month_rows} hourly rows")
    time.sleep(1)  

print(f"\nTotal raw rows collected: {len(all_records)}")

# --- To DataFrame (UTC, canonical schema) ------------------------------------
df = pd.DataFrame(all_records)
if df.empty:
    raise RuntimeError("No rows collected from Elhub for 2021 monthly windows.")

df["startTime"]   = pd.to_datetime(df["startTime"], utc=True, errors="coerce")
df["quantityKwh"] = pd.to_numeric(df["quantityKwh"], errors="coerce")

# Hard filter to 2021 in UTC (guard against API drift)
start_2021 = pd.Timestamp("2021-01-01 00:00:00+00:00")
end_2021   = pd.Timestamp("2021-12-31 23:59:59.999999+00:00")
before = len(df)
df_final = df[
    (df["startTime"] >= start_2021) & (df["startTime"] <= end_2021)
][["priceArea", "productionGroup", "startTime", "quantityKwh"]].copy()
after = len(df_final)

df_final.sort_values(["priceArea", "startTime"], inplace=True, kind="stable")
df_final.reset_index(drop=True, inplace=True)

print(f"[Filter] kept {after} of {before} rows in 2021; dropped {before-after}.")
if not df_final.empty:
    print("Date span:", df_final["startTime"].min(), "→", df_final["startTime"].max())
    print("Areas:", df_final["priceArea"].nunique(),
          "| Groups:", df_final["productionGroup"].nunique(),
          "| Rows:", len(df_final))
else:
    print("⚠ No 2021 rows remained. Check endpoint/params or load local 2021 cache.")

# Safe to inspect
df_final.head(10)



Requesting Elhub with params: {'dataset': 'PRODUCTION_PER_GROUP_MBA_HOUR', 'startTimeFrom': '2021-01-01T00:00:00Z', 'startTimeTo': '2021-12-31T23:59:00Z'}
…did not look like 2021; trying next param style.
Requesting Elhub with params: {'dataset': 'PRODUCTION_PER_GROUP_MBA_HOUR', 'fromTime': '2021-01-01T00:00:00Z', 'toTime': '2021-12-31T23:59:00Z'}
…did not look like 2021; trying next param style.
Requesting Elhub with params: {'dataset': 'PRODUCTION_PER_GROUP_MBA_HOUR', 'startTime': '2021-01-01T00:00:00Z', 'endTime': '2021-12-31T23:59:00Z'}
…did not look like 2021; trying next param style.
⚠ API ignored all bound variants; will hard-filter after normalization.
Price areas returned: 6
Prepared hourly rows: 18989


,priceArea,productionGroup,startTime,quantityKwh
0,NO1,hydro,2025-10-13T23:00:00+02:00,2121128.5
1,NO1,hydro,2025-10-14T00:00:00+02:00,2204685.8
2,NO1,hydro,2025-10-14T01:00:00+02:00,2179913.0


In [ ]:
from pyspark.sql import functions as F

# Spark DataFrame with lowercase Cassandra column names
df_spark = spark.createDataFrame(df_pd.rename(columns={
    "priceArea":"pricearea",
    "productionGroup":"productiongroup",
    "startTime":"starttime",
    "quantityKwh":"quantitykwh",
}))

# Append into elhub.production_hourly
df_spark.write.format("org.apache.spark.sql.cassandra") \
    .options(keyspace="elhub", table="production_hourly") \
    .mode("append").save()

# Small confirmation
total_rows = (spark.read.format("org.apache.spark.sql.cassandra")
              .options(keyspace="elhub", table="production_hourly").load().count())
print("Cassandra row count now:", total_rows)


In [ ]:
# Ensure plots render inline in Jupyter
try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

import matplotlib.pyplot as plt
from pathlib import Path

# Read back from Cassandra for plotting (as requested in assignment)
prod = (spark.read.format("org.apache.spark.sql.cassandra")
        .options(keyspace="elhub", table="production_hourly").load()
        .select("pricearea","productiongroup","starttime","quantitykwh"))

# Limit strictly to YEAR to avoid mixing if you run multiple years later
prod = prod.filter(F.year("starttime")==YEAR)

# ----- Pie: total production of the year by group for chosen price area -----
totals = (prod.filter(F.col("pricearea")==CHOSEN_AREA)
          .groupBy("productiongroup")
          .agg(F.sum("quantitykwh").alias("sumKwh"))
          .orderBy(F.desc("sumKwh"))).toPandas()

plt.figure()
if not totals.empty:
    plt.pie(totals["sumKwh"], labels=totals["productiongroup"], autopct="%1.1f%%", startangle=90)
    plt.title(f"Total production {YEAR} — {CHOSEN_AREA}")
else:
    plt.text(0.5,0.5,"No data", ha="center")
plt.show()

# ----- Line: first month (January) with separate lines per group -----
jan = (prod.filter((F.col("pricearea")==CHOSEN_AREA) &
                   (F.col("starttime") >= F.to_timestamp(F.lit(f"{YEAR}-01-01T00:00:00Z"))) &
                   (F.col("starttime") <  F.to_timestamp(F.lit(f"{YEAR}-02-01T00:00:00Z"))))
       .groupBy("productiongroup","starttime")
       .agg(F.sum("quantitykwh").alias("kWh"))).toPandas()

plt.figure()
if not jan.empty:
    pvt = jan.pivot(index="starttime", columns="productiongroup", values="kWh").sort_index()
    for col in pvt.columns:
        plt.plot(pvt.index, pvt[col], label=col)
    plt.legend(); plt.title(f"Hourly production — Jan {YEAR} — {CHOSEN_AREA}")
    plt.xlabel("Time (UTC)"); plt.ylabel("kWh"); plt.tight_layout()
else:
    plt.text(0.5,0.5,"No January data", ha="center")
plt.show()


In [ ]:
if not MONGO_URI:
    print("⚠️  MONGO_URI is not set; skipping Atlas export. Set it and re-run this cell.")
else:
    from pymongo import MongoClient, ASCENDING

    # Connect to Atlas using SRV URI from env
    client = MongoClient(MONGO_URI)
    db = client["energy"]  # choose DB explicitly
    client.admin.command("ping")
    print("Connected to", db.name)
    coll = db["elhub_production_2021"]  # collection used by the Streamlit page

    # Clean + index for fast lookups
    coll.delete_many({})
    coll.create_index([("priceArea", ASCENDING), ("productionGroup", ASCENDING), ("startTime", ASCENDING)])

    # Convert timestamps to string UTC (ISO Z) for portability
    df_out = df_pd.copy()
    df_out["startTime"] = df_out["startTime"].dt.tz_convert("UTC").dt.strftime("%Y-%m-%dT%H:%M:%SZ")

    # Insert in batches to avoid payload limits
    BATCH = 200_000
    inserted = 0
    for i in range(0, len(df_out), BATCH):
        batch = df_out.iloc[i:i+BATCH]
        if not batch.empty:
            coll.insert_many(batch.to_dict(orient="records"))
            inserted += len(batch)

    print(f"Inserted {inserted} documents into Atlas → {coll.full_name}")
